In [3]:
%pip install geopandas matplotlib seaborn mapclassify folium

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    python-xyz', where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Arch-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-Arch packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

ModuleNotFoundError: No module named 'geopandas'

In [ ]:
# --- CONFIGURATION ---
WARDS_FILE = "data/interim/vadodara_project.gpkg"
ENV_FILE = "data/processed/ward_indicators.csv"
ACCESS_FILE = "data/processed/ward_travel_times.csv"

In [ ]:
# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

In [ ]:
# 1. LOAD DATA
print("-> Loading Data Layers...")
wards = gpd.read_file(WARDS_FILE)
env_data = pd.read_csv(ENV_FILE)
access_data = pd.read_csv(ACCESS_FILE)

In [ ]:
# 2. MERGE LAYERS
# We combine the Map + Crowding Data + Travel Time Data
final_df = wards.merge(env_data, on='ward_id').merge(access_data, on='ward_id')

In [ ]:
# 3. CALCULATE "URBAN OPPORTUNITY INDEX"
# Methodology:
# - Normalize all columns to 0-1 range.
# - Invert "Negative" metrics (where High = Bad).

def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

# A. Normalize Basic Columns
final_df['n_green'] = normalize(final_df['green_density'])         # High = Good
final_df['n_built'] = normalize(final_df['building_density'])      # High = Bad (Crowded)
final_df['n_hosp'] = normalize(final_df['hospitals_min'])          # High = Bad (Far)
final_df['n_school'] = normalize(final_df['schools_min'])          # High = Bad (Far)
final_df['n_trans'] = normalize(final_df['transport_min'])         # High = Bad (Far)

In [ ]:
# B. Calculate Sub-Indices (Higher Score = Better Life)
# Environment Score: High Greenery + Low Crowding
final_df['Score_Env'] = (final_df['n_green'] + (1 - final_df['n_built'])) / 2

In [ ]:
# Access Score: Short times to Hospital, School, Transport
final_df['Score_Access'] = ((1 - final_df['n_hosp']) + (1 - final_df['n_school']) + (1 - final_df['n_trans'])) / 3

In [ ]:
# C. FINAL INDEX (The "Inequality Score")
final_df['Final_Index'] = (final_df['Score_Env'] + final_df['Score_Access']) / 2

print("-> Index Calculated. Summary:")
print(final_df[['ward_id', 'Final_Index', 'hospitals_min', 'green_density']].head())

In [ ]:
# 4. VISUALIZATION (The Dashboard)
fig, axes = plt.subplots(2, 2, figsize=(18, 16))
fig.suptitle('Urban Inequality in Vadodara: Multidimensional Analysis', fontsize=24, weight='bold')

In [ ]:
# Helper function for consistent mapping
def plot_map(ax, column, title, cmap, label):
    final_df.plot(column=column, cmap=cmap, linewidth=0.8, edgecolor='0.8', legend=True, ax=ax, legend_kwds={'label': label, 'orientation': "horizontal"})
    ax.set_title(title, fontsize=14, weight='bold')
    ax.axis('off')

In [ ]:
# Map 1: The Master Index (Inequality)
# Light Yellow = High Opportunity, Dark Purple = Neglected
plot_map(axes[0, 0], 'Final_Index', 'Overall Urban Opportunity Index\n(Brighter = Better Quality of Life)', 'viridis', "Index Score (0-1)")

In [ ]:
# Map 2: Medical Access
# Red/Dark = Far from Hospital
plot_map(axes[0, 1], 'hospitals_min', 'Medical Accessibility\n(Travel Time in Minutes)', 'Reds', "Minutes to Nearest Hospital")

In [ ]:
# Map 3: Green Spaces
# Green = High Greenery
plot_map(axes[1, 0], 'green_density', 'Green Lung Cover\n(% of Ward Area)', 'Greens', "Percentage Green Space")

In [ ]:
# Map 4: Transport Access
# Blue = Far from Transport
plot_map(axes[1, 1], 'transport_min', 'Public Transport Access\n(Travel Time in Minutes)', 'Blues', "Minutes to Bus/Train")

In [ ]:
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust for suptitle

In [ ]:
# Save to file
output_path = "results/maps/thesis_dashboard.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\n✅ SUCCESS: Dashboard saved to {output_path}")
plt.show()